
# Fine-tune IndoBERT-p1 pada Mafindo-2024

Menjalankan `classification/finetune_indobert.py`

Folder Drive `hoax classification` berisi:
- `finetune_indobert.py`, `cross_dataset.py`, `short_claims_probe.py` (dari `classification/`) dan `mendeley_audit.py` (dari `classification/data_audit/`)
- `600 news with valid hoax label.csv` dan `600 news labelling process.xlsx`
- `short_claims.jsonl` (opsional, dari `data/interim/classifier/`; hasil probe, teks pihak ketiga)

## 1. Setup

In [1]:
!nvidia-smi -L
!pip -q install datasets openpyxl
import torch; print("torch", torch.__version__, "| cuda", torch.cuda.is_available())

GPU 0: Tesla T4 (UUID: GPU-129c4f85-93d7-ce76-ca5f-ba23a8097acf)
torch 2.14.1+cu130 | cuda True


## 2. Connect Drive dan folder

In [2]:
import os, shutil
from google.colab import drive
drive.mount("/content/drive")
DRIVE_DIR = "/content/drive/MyDrive/SEMESTER 7/NLP/Tubes"
OUT_DIR = DRIVE_DIR + "/indobert_results"

dest = {
    "finetune_indobert.py": "classification",
    "cross_dataset.py": "classification",
    "short_claims_probe.py": "classification",
    "mendeley_audit.py": "classification/data_audit",
    "600 news with valid hoax label.csv": "data/raw/mendeley",
    "600 news labelling process.xlsx": "data/raw/mendeley",
    "short_claims.jsonl": "data/interim/classifier",
}
for name, folder in dest.items():
    os.makedirs(folder, exist_ok=True)
    src = os.path.join(DRIVE_DIR, name)
    if os.path.exists(src):
        shutil.copy(src, os.path.join(folder, name))
need = [k for k in dest if k != "short_claims.jsonl"]
missing = [k for k in need if not os.path.exists(os.path.join(dest[k], k))]
print("Belum ada:" if missing else "Berkas wajib lengkap.", missing or "")
print("probe tersedia:", os.path.exists("data/interim/classifier/short_claims.jsonl"))

Mounted at /content/drive
Berkas wajib lengkap. 
probe tersedia: False


## 3. Uji alur: data kecil, 1 epoch, 1 seed

In [3]:
!python classification/finetune_indobert.py --data-dir data/raw/mendeley --probe data/interim/classifier/short_claims.jsonl --smoke --out-dir results/indobert

device cuda | model indobenchmark/indobert-base-p1 | max_length 128 | epochs 1 | seeds [13]
README.md: 100% 858/858 [00:00<00:00, 5.08MB/s]

train.csv: downloading bytes:  34% 12.0M/35.1M [00:00<00:01, 18.8MB/s,  264kB/s  ]
train.csv: downloading bytes: 100% 20.9M/20.9M [00:00<00:00, 21.8MB/s, 2.04MB/s  ]
train.csv: reconstructing file: 100% 35.1M/35.1M [00:00<00:00, 36.6MB/s, 3.43MB/s  ]

test.csv: downloading bytes:  11% 988k/8.61M [00:00<00:04, 1.84MB/s]
test.csv: downloading bytes: 100% 5.14M/5.14M [00:00<00:00, 7.58MB/s,  505kB/s  ]
test.csv: reconstructing file: 100% 8.61M/8.61M [00:00<00:00, 12.7MB/s,  847kB/s  ]
Generating train split: 100% 23641/23641 [00:00<00:00, 29502.29 examples/s]
Generating test split: 100% 5911/5911 [00:00<00:00, 27923.32 examples/s]
config.json: 100% 1.53k/1.53k [00:00<00:00, 4.46MB/s]
tokenizer_config.json: 100% 2.00/2.00 [00:00<00:00, 7.14kB/s]
vocab.txt: 100% 229k/229k [00:00<00:00, 17.8MB/s]
special_tokens_map.json: 100% 112/112 [00:00<00:00, 378kB

## 4. Run penuh (3 seed, 3 epoch, panjang 256)

In [4]:
!python classification/finetune_indobert.py --data-dir data/raw/mendeley     --probe data/interim/classifier/short_claims.jsonl     --out-dir "$OUT_DIR"

device cuda | model indobenchmark/indobert-base-p1 | max_length 256 | epochs 3 | seeds [13, 42, 2026]
train 21276 | val 2365 | mafindo test 5911 | mendeley 600
[transformers] You passed `num_labels=2` which is incompatible to the `id2label` map of length `5`.
Loading weights: 100% 199/199 [00:00<00:00, 19839.00it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: indobenchmark/indobert-base-p1
Key               | Status  | 
------------------+---------+-
classifier.weight | MISSING | 
classifier.bias   | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
/content/classification/finetune_indobert.py:118: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /__w/pytorch/pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:820.)
  losses.append(float(loss))


## 5. Summary

In [5]:
import json
s = json.load(open(OUT_DIR + "/summary.json"))
print(s["meta"])
for k, v in s["aggregate"].items():
    print(k, {m: (v[m]["mean"], v[m]["sd"]) for m in v})

{'model': 'indobenchmark/indobert-base-p1', 'seeds': [13, 42, 2026], 'epochs': 3, 'batch_size': 32, 'lr': 2e-05, 'max_length': 256, 'text_field': 'content', 'threshold': 0.5, 'val_split': '10% stratified, seed 13', 'smoke': False, 'torch': '2.14.1+cu130'}
mafindo_test {'macro_f1': (0.987, 0.001), 'auroc': (0.998, 0.0), 'hoax_precision': (0.984, 0.001), 'hoax_recall': (0.988, 0.002)}
mafindo_test_titles_not_in_train {'macro_f1': (0.987, 0.001), 'auroc': (0.998, 0.0), 'hoax_precision': (0.984, 0.001), 'hoax_recall': (0.988, 0.002)}
mendeley_all {'macro_f1': (0.524, 0.029), 'auroc': (0.585, 0.01), 'hoax_precision': (0.412, 0.031), 'hoax_recall': (0.429, 0.12)}
mendeley_no_debunk {'macro_f1': (0.459, 0.031), 'auroc': (0.498, 0.007), 'hoax_precision': (0.56, 0.022), 'hoax_recall': (0.43, 0.132)}
mendeley_debunk_only {'macro_f1': (0.526, 0.012), 'auroc': (0.603, 0.019), 'hoax_precision': (0.245, 0.018), 'hoax_recall': (0.427, 0.089)}


## 6. Model untuk demo CLI (1 seed, simpan checkpoint)

In [ ]:
DEMO_DIR = DRIVE_DIR + "/indobert_demo"
!python classification/finetune_indobert.py --data-dir data/raw/mendeley     --probe data/interim/classifier/short_claims.jsonl     --seeds 13 --out-dir "$DEMO_DIR" --save-dir "$DEMO_DIR/model"
!ls "$DEMO_DIR/model/seed13"